In [15]:
# !pip install financedatabase -U

In [16]:
import pandas as pd
import os, glob, re, sys
import financedatabase as fd

INPUT_FILE = r"../data/interim/01b_data_after_yfinance.csv"
OUTPUT_FILE = r"../data/interim/01c_data_after_financedatabase.csv"

equities = fd.Equities()
equities.select()

equities_df = pd.DataFrame(equities.data)


In [17]:
# Só reseta o índice se ele ainda não foi resetado (ou seja, se
# ainda tem nome — RangeIndex padrão tem index.names == [None])
if equities_df.index.names != [None]:
    equities_df = equities_df.reset_index()

# Renomeia só se a coluna "symbol" ainda existir (evita KeyError
# se essa célula já rodou antes e "symbol" já virou "ticker_symbol")
if "symbol" in equities_df.columns:
    equities_df = equities_df.rename(columns={"symbol": "ticker_symbol"})

equities_df = equities_df.drop(columns=[
    col for col in [
        "level_0", "index", "name", "summary", "currency",
        "exchange", "mic", "market", "country", "state",
        "city", "zipcode", "market_cap", "isin"
    ] if col in equities_df.columns
])

display(equities_df.head(10))

for col in equities_df.columns:
    print(col)

,ticker_symbol,sector,industry_group,industry,website,cusip,figi,composite_figi,shareclass_figi,delisted
0,000002.SZ,Real Estate,Real Estate,Real Estate Management & Development,http://www.vanke.com,NaN,NaN,BBG006KY4KD6,BBG006KY4KG3,False
1,000004.SZ,Financials,Diversified Financials,Diversified Financial Services,http://www.sz000004.cn,NaN,BBG000BZFTX3,BBG000BZFT18,BBG001SJWNX3,False
2,000005.SZ,Real Estate,Real Estate,Real Estate Management & Development,http://www.fountain.com.cn,NaN,NaN,NaN,NaN,False
3,000006.SZ,Real Estate,Real Estate,Real Estate Management & Development,http://www.zhenye.com,NaN,BBG000BZFRZ5,BBG000BZFR12,BBG001SJWNW4,False
4,000007.SZ,Consumer Discretionary,Consumer Services,"Hotels, Restaurants & Leisure",http://www.sz000007.com,NaN,BBG000BZDNB2,BBG000BZDMM2,BBG001SJWM10,False
5,000009.SZ,Industrials,Capital Goods,Industrial Conglomerates,http://www.chinabaoan.com,NaN,BBG000BZDLT7,BBG000BZDL53,BBG001SJWM01,False
6,000010.SZ,Industrials,Capital Goods,Construction & Engineering,http://www.eco-beauty.cn,NaN,BBG000P6HP77,BBG000P6HN18,BBG001SJWLZ5,False
7,000014.SZ,Real Estate,Real Estate,Real Estate Management & Development,http://www.shahe.cn,NaN,BBG000BZBX31,BBG000BZBV44,BBG001SJWDP4,False
8,000016.SZ,Information Technology,Technology Hardware & Equipment,"Electronic Equipment, Instruments & Components",http://www.konka.com,NaN,BBG000BZCNJ5,BBG000BZCMX1,BBG001SJWHW7,False
9,000017.SZ,Consumer Discretionary,Consumer Services,"Hotels, Restaurants & Leisure",http://cbcbicycle.diytrade.com/sdp/628227/4/ho...,NaN,BBG000BZDHB5,BBG000BZDGS9,BBG001SJWLV9,False


ticker_symbol
sector
industry_group
industry
website
cusip
figi
composite_figi
shareclass_figi
delisted


In [18]:
import pandas as pd

yfinance_df = pd.read_csv(INPUT_FILE)

unresolved_df = yfinance_df[yfinance_df["website"].isnull()]

print(f"Number of unresolved tickers: {len(unresolved_df)}")

display(unresolved_df.head(10))

for ticker in unresolved_df["ticker_symbol"]:
    if ticker in equities_df["ticker_symbol"].values:
        print(f"Found website for {ticker}")
        print(f"Website: {equities_df[equities_df['ticker_symbol'] == ticker]['website'].values[0]}")
        unresolved_df.loc[unresolved_df["ticker_symbol"] == ticker, "website"] = equities_df[equities_df["ticker_symbol"] == ticker]["website"].values[0]

display(unresolved_df.head(100))

print(f"Number of unresolved tickers after update: {len(unresolved_df[unresolved_df['website'].isnull()])}")

Number of unresolved tickers: 514


,ticker_symbol,rank,company_name,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
407,MCHA.DE,408,Michelin,30533561783,36.947600,France,NaN,NaN,NaN,NaN,NaN,NaN,error: website not found
621,FISV,622,Fiserv,21090000000,53.000000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: website not found
916,0489.HK,917,Dongfeng Motor,14551321587,1.218500,China,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
1146,XEKT.MC,1147,Grupo Elektra,11317736000,20.245800,Mexico,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
1226,7205.T,1227,Hino Motors,10354593406,2.429610,Japan,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
1671,SGH.AX,1672,Seven Group Holdings\r\n (SGH),7067162150,27.669700,Australia,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
1993,SEM,1992,Select Medical Holdings,5521134000,16.510000,United States,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
2007,WG.L,2006,John Wood Group,5478000000,0.399424,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
2191,YELLQ,2190,Yellow Corporation,4846000000,0.010200,United States,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
2204,US30212W1009.SG,2203,eXp World Holdings,4822946000,6.740000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: website not found


Found website for FISV
Website: nan
Found website for XEKT.MC
Website: http://www.grupoelektra.com.mx
Found website for 7205.T
Website: http://www.hino.co.jp
Found website for SGH.AX
Website: http://www.slatergordon.com.au
Found website for SEM
Website: http://www.selectmedical.com
Found website for WG.L
Website: http://www.woodplc.com
Found website for US30212W1009.SG
Website: nan
Found website for 7105.T
Website: http://www.logisnext.com
Found website for 2692.T
Website: http://www.itochu-shokuhin.com
Found website for FDM.MU
Website: nan
Found website for UNBL.PA
Website: http://unibel.fr
Found website for 9719.T
Website: http://www.scsk.jp
Found website for STLN.SW
Website: http://www.swisssteel-group.com
Found website for VTNR
Website: http://www.vertexenergy.com
Found website for SYAB.VI
Website: http://www.synlab.com
Found website for LION
Website: nan
Found website for BAW.JO
Website: http://www.barloworld.com
Found website for 086520.KQ
Website: http://www.ecopro.co.kr
Found w

,ticker_symbol,rank,company_name,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
407,MCHA.DE,408,Michelin,30533561783,36.947600,France,NaN,NaN,NaN,NaN,NaN,NaN,error: website not found
621,FISV,622,Fiserv,21090000000,53.000000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: website not found
916,0489.HK,917,Dongfeng Motor,14551321587,1.218500,China,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
1146,XEKT.MC,1147,Grupo Elektra,11317736000,20.245800,Mexico,http://www.grupoelektra.com.mx,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
1226,7205.T,1227,Hino Motors,10354593406,2.429610,Japan,http://www.hino.co.jp,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6998,KGL.KW,6996,Kuwait and Gulf Link Transport Company,353507777,234.910000,Kuwait,NaN,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
7022,SOR.OL,7020,Sparebanken Sør,349148887,20.163100,Norway,http://www.sor.no,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
7070,FARM,7068,Farmer Brothers,337719000,1.280000,United States,http://www.farmerbros.com,NaN,NaN,NaN,NaN,NaN,not_found_yf_search
7087,ALB.MC,7085,Corporación Financiera Alba,334650449,97.951500,Spain,http://www.corporacionalba.es,NaN,NaN,NaN,NaN,NaN,not_found_yf_search


Number of unresolved tickers after update: 355


In [19]:
import pandas as pd

current_df = pd.read_csv(INPUT_FILE)

current_df.update(unresolved_df)

current_df.to_csv(OUTPUT_FILE)

In [20]:
# save csv with the ones that dont have website yet (with "ticker_symbol", "company_name", "country", "website" columns)

missing_website_df = unresolved_df[unresolved_df["website"].isnull()][["ticker_symbol", "company_name", "company_country", "website"]]
missing_website_df.to_csv("missing_websites.csv", index=False)


In [21]:
# create cache of known websites to avoid re-scraping the same companies again and again
# data to store in cache: ticker_symbol, company_name, company_country, website, sector, industry, country

# create function to create or update the cache

def update_cache(ticker_symbol, company_name, company_country, website, sector, industry, country):
    cache_file = "company_data_cache.csv"
    new_data = pd.DataFrame([{
        "ticker_symbol": ticker_symbol,
        "company_name": company_name,
        "company_country": company_country,
        "website": website,
        "sector": sector,
        "industry": industry,
        "country": country
    }])
    
    if os.path.exists(cache_file):
        cache_df = pd.read_csv(cache_file)
        cache_df = pd.concat([cache_df, new_data], ignore_index=True)
    else:
        cache_df = new_data
    
    # drop duplicates based on ticker_symbol, keeping the last occurrence
    cache_df = cache_df.drop_duplicates(subset=["ticker_symbol"], keep="last")

    cache_df.to_csv("../data/interim/" + cache_file, index=False)

# run the update_cache function for each row in the equities_df DataFrame
for index, row in equities_df.iterrows():
    update_cache(
        ticker_symbol=row["ticker_symbol"],
        company_name=row["company_name"],
        company_country=row["company_country"],
        website=row["website"],
        sector=row["sector"],
        industry=row["industry"],
        country=row["country"]
    )


KeyError: 'company_name'